# Vela Kp Forecast Validation
This notebook validates the Phase 3 LightGBM Space Weather Forecast Model against Solar Cycle 25 (2020–2025) test data.


In [ ]:
import sys
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import mean_squared_error, f1_score

sys.path.insert(0, str(Path.cwd().parent))
try:
    from ingestion.omni_client import OMNIClient
    from modeling.feature_engineering import build_features
except ImportError:
    # Handle if running from vela root
    sys.path.insert(0, str(Path.cwd()))
    from ingestion.omni_client import OMNIClient
    from modeling.feature_engineering import build_features

sns.set_theme(style="darkgrid")
plt.rcParams['figure.figsize'] = (12, 6)


In [ ]:
# 1. Load trained model
model_path = Path("modeling/model_registry/kp_baseline_lgbm.pkl")
if not model_path.exists():
    model_path = Path("../modeling/model_registry/kp_baseline_lgbm.pkl")

with open(model_path, "rb") as f:
    artifact = pickle.load(f)

model = artifact["model"]
feature_names = artifact["feature_names"]
train_cutoff = artifact.get("train_cutoff", "2020-01-01")
print(f"Loaded model trained up to: {train_cutoff}")


In [ ]:
# 2. Load OMNI test data
client = OMNIClient()
df = client.parse_all(2020, 2025)
X, y = build_features(df, target_horizon_hours=3)

# Test set only (Solar Cycle 25)
test_mask = X.index >= pd.Timestamp('2020-01-01', tz='UTC')
X_test = X[test_mask]
y_test = y[test_mask]

print(f"Test set: {len(X_test)} samples from {X_test.index.min()} to {X_test.index.max()}")


In [ ]:
# Generate predictions
y_pred = model.predict(X_test)
test_rmse = np.sqrt(mean_squared_error(y_test, y_pred))
print(f"Overall Test RMSE: {test_rmse:.4f}")


## Plot 1: Predicted vs. Actual Kp Scatter Plot

In [ ]:
# Plot 1: Scatter plot
colors = np.where(y_test >= 5, 'red', np.where(y_test >= 3, 'orange', 'green'))

plt.figure(figsize=(10, 8))
plt.scatter(y_test, y_pred, alpha=0.1, c=colors, s=10)
plt.plot([0, 9], [0, 9], 'k--', lw=2)
plt.xlabel("Actual Kp")
plt.ylabel("Predicted Kp")
plt.title("Predicted vs Actual Kp (t+3h) - Solar Cycle 25")
plt.xlim(0, 9)
plt.ylim(0, 9)
plt.tight_layout()
plt.show()


## Plot 2: Time Series Window of Strongest Storm

In [ ]:
# Find peak Kp in test set
peak_idx = y_test.argmax()
peak_time = y_test.index[peak_idx]

# 30-day window centered on peak
window_start = peak_time - pd.Timedelta(days=15)
window_end = peak_time + pd.Timedelta(days=15)
mask = (y_test.index >= window_start) & (y_test.index <= window_end)

plt.figure(figsize=(14, 6))
plt.plot(y_test[mask].index, y_test[mask].values, label="Actual Kp", color="black", lw=2)
plt.plot(y_test[mask].index, y_pred[mask], label="Predicted Kp (Vela)", color="dodgerblue", alpha=0.8, lw=2)
plt.axhline(5.0, color='red', linestyle='--', alpha=0.5, label="Storm Threshold (G1)")
plt.xlabel("Date")
plt.ylabel("Kp Index")
plt.title(f"30-Day Window: Strongest Storm (Peak: {peak_time.date()})")
plt.legend()
plt.tight_layout()
plt.show()


## Plot 3: Reliability Diagram (Calibration Plot)

In [ ]:
# Bin predicted values
bins = np.arange(0, 10, 1)
bin_indices = np.digitize(y_pred, bins)

bin_means_pred = [y_pred[bin_indices == i].mean() if len(y_pred[bin_indices == i]) > 0 else np.nan for i in range(1, len(bins))]
bin_means_actual = [y_test[bin_indices == i].mean() if len(y_test[bin_indices == i]) > 0 else np.nan for i in range(1, len(bins))]

plt.figure(figsize=(8, 8))
plt.plot(bin_means_pred, bin_means_actual, marker='o', linestyle='-', lw=2)
plt.plot([0, 9], [0, 9], 'k--', lw=2, label="Perfect Calibration")
plt.xlabel("Mean Predicted Kp (by bin)")
plt.ylabel("Mean Actual Kp")
plt.title("Reliability Diagram (Calibration Plot)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## Plot 4: RMSE by Kp Bin

In [ ]:
# RMSE by actual Kp bin
bin_rmse = []
bin_labels = []
for i in range(0, 8):
    if i == 7:
        mask = y_test >= 7
        label = "7+"
    else:
        mask = (y_test >= i) & (y_test < i+1)
        label = f"{i}-{i+1}"
    
    if mask.sum() > 0:
        rmse = np.sqrt(mean_squared_error(y_test[mask], y_pred[mask]))
    else:
        rmse = 0
    bin_rmse.append(rmse)
    bin_labels.append(label)

plt.figure(figsize=(10, 6))
bars = plt.bar(bin_labels, bin_rmse, color='teal')
plt.xlabel("Actual Kp Bin")
plt.ylabel("RMSE")
plt.title("RMSE Breakdown by Actual Kp Bin")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.05, f"{yval:.2f}", ha='center', va='bottom')
plt.tight_layout()
plt.show()


## Plot 5: Comparison Benchmark

In [ ]:
# Extract persistence predictions (using lag feature or actual previous Kp)
# The current Kp is in X_test['Kp']
pers_pred = X_test['Kp'].values
pers_rmse = np.sqrt(mean_squared_error(y_test, pers_pred))

benchmark_rmse = 0.64 # Published NN benchmark at 3h

metrics = ["Vela LightGBM Baseline", "Persistence (Current Kp)", "Published NN Benchmark"]
rmse_vals = [test_rmse, pers_rmse, benchmark_rmse]

plt.figure(figsize=(10, 6))
bars = plt.bar(metrics, rmse_vals, color=['dodgerblue', 'gray', 'purple'])
plt.ylabel("RMSE (Lower is better)")
plt.title("Performance Comparison (3-Hour Horizon)")
for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 0.02, f"{yval:.2f}", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()


## Claim Boundaries

### What Vela MAY claim
- Satellite positions are propagated from TLE data updated approximately 3 times daily from the 18th Space Defense Squadron via CelesTrak
- The Kp forecast is produced by a LightGBM model trained on NASA OMNI hourly solar wind data from 1996 to 2025, validated on Solar Cycle 25 (2020–present)
- Validated test RMSE: 0.77 at 3-hour horizon on Solar Cycle 25 test set
- Validated storm detection F1: 0.55 for Kp ≥ 5 events
- The model improves on naive persistence baseline by 20.7% (skill score 0.206)
- NEO close approach data and impact probabilities are sourced directly from NASA JPL Sentry-II and CNEOS — Vela does not independently compute orbital uncertainty
- Launch history and geopolitical data are sourced from GCAT (Jonathan McDowell), UCS Satellite Database (7,560 active satellites, 99.9% join rate), and Launch Library 2
- Commercial payload dominance: 80.4% of active satellites are commercial operators

### What Vela MAY NOT claim
- That satellite positions are exact real-time GPS coordinates — they are SGP4 propagations from TLEs with inherent positional uncertainty
- That the Kp forecast meets or exceeds the published neural network benchmark of RMSE 0.64 — the LightGBM baseline does not
- That Vela independently computes asteroid impact probability
- That conjunction risk data constitutes operational collision avoidance intelligence
- That the Kp forecast is an operational space weather warning system
